# Multi-Task Fine-Tuning (MTFT)
### Advanced Fine-Tuning Paradigms  ·  Colab T4 (16 GB) ready

> **Instruction Tuning** (previous notebook) mixes many tasks to maximise **breadth** — there is no single target task, and the metric is zero-shot generalisation.
> **MTFT has a designated target task** it must be excellent at, and blends in other data whose only job is to **stop the model forgetting what it already knew**. Same machinery, opposite optimisation goal.
> The deliverable of an MTFT project is therefore never one model — it is a **Pareto frontier**: target-task gain on one axis, retained general capability on the other. This notebook trains **three** models at three mixture ratios and tabulates that frontier.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **MTFT** optimises a **weighted sum of per-task objectives simultaneously**, sampling each task's data from its own distribution within every batch:
  $$\mathcal{L}_{\text{MTFT}}(\theta) = \sum_{i=1}^{T} w_i \cdot \mathbb{E}_{(x,y)\sim \mathcal{D}_i}\big[\ell_i(\theta; x, y)\big], \qquad \text{examples drawn with sampling probability } p_i$$
- **`p_i` and `w_i` are two independent dials, and conflating them is the most common MTFT implementation error:**
  - **`p_i` (sampling ratio)** changes the *empirical distribution* — how often task `i` appears in a batch.
  - **`w_i` (loss weight)** changes the *gradient magnitude* of task `i` when it does appear.
  - They are equivalent only in the expectation of the **mean** gradient. They are **not** equivalent under Adam, which normalises per-coordinate by a running second moment: a task that appears rarely but with a large weight produces spiky, high-variance updates, while the same task sampled more often at weight 1.0 produces a smooth one. Section 3 exposes both as separate arguments.
- **Lineage:** **MT-DNN** (*Liu et al., 2019*) for joint multi-task training of a shared encoder; **T5** (*Raffel et al., 2020*) for examples-proportional mixing with a cap; **MUPPET** (*Aghajanyan et al., 2021*) for pre-finetuning across ~50 tasks and the observation that **too few tasks actively hurt** while many tasks help; and the **rehearsal/replay** line from continual learning (*Robins, 1995*; *Kirkpatrick et al., 2017* for EWC; *Ibrahim et al., 2024* for the modern LLM recipe).
- **Catastrophic forgetting, stated precisely:** single-task fine-tuning minimises `L_target` under **no constraint** on `L_general`, so gradient descent is free to walk into a basin where `L_general` is arbitrarily high. MTFT converts a **sequential** problem into a **joint** one, which keeps a non-zero gradient component on `L_general` at *every* step. This is mechanically identical to CPT's replay term, generalised to `n` tasks and `n` objectives.
- **"Varied objective functions"** in a single decoder-only run means, in practice, three concrete things:
  1. **Per-example loss weighting keyed by a `task_id`** (implemented in Section 3),
  2. **Per-task masking policy** — e.g. completion-masked for chat, full-sequence for a raw-text anchor,
  3. Genuinely different losses/heads (CLM + classification + ranking), which needs a custom head per task and is the same idea one layer up.
- **The trap that eats most MTFT runs — token-mean vs. example-mean loss.** The default HF loss averages cross-entropy over **all supervised tokens in the batch**. If your target task's completions are ~15 tokens (SQL) and your anchor task's are ~200 (chat), then a configured **50/50 sampling ratio is a ~7 / 93 gradient ratio**. The mixture you configured is not the mixture you got. Section 3 measures this gap on a real batch, then fixes it by averaging **per example** first.

### One-sentence definition of the mechanics

> **MTFT jointly minimises a weighted sum of several tasks' losses — with sampling ratios and loss weights as separate, explicitly-controlled dials — so that adapting hard to a narrow target task cannot walk the weights into a region where the model's pre-existing general capabilities have collapsed.**

### The exact engineering problem it solves

- **Narrow fine-tuning silently destroys general capability, and you will not notice from the loss curve.** Train an instruction-following model exclusively on text-to-SQL and the SQL metric climbs beautifully while the model progressively loses the ability to hold a conversation, follow an unrelated instruction, or refuse anything. The training loss reports **nothing** about this — it is measured only on the target distribution.
- **Sequential fine-tuning has no mechanism to prevent it.** Regularisation toward the old weights (L2-SP, EWC) helps a little and costs a lot of engineering; **rehearsing the old data is far cheaper and works far better.**
- **Multiple deployed capabilities cannot be `n` separate LoRAs when they must compose.** Adapter-per-task means a routing decision at inference, `n` adapter loads, and no ability to answer a request that needs two skills at once. MTFT produces **one** set of weights that holds all of them.
- **It is also how you avoid negative transfer**: tasks trained jointly can interfere (conflicting gradients) or reinforce (shared structure). You only find out which by mixing them and measuring — which is exactly why the method is a **sweep**, not a single run.
- **What MTFT does *not* do:** it does not add knowledge (that is CPT) and it does not create an instruction interface from nothing (that is IT). It **preserves** capabilities while you specialise. Start from a model that already has the capability you are trying to keep — this notebook starts from **`-Instruct`** for exactly that reason.

---

### The Human Element — Hugging Face datasets for MTFT

| HF path | What it is | Why it's structured this way for MTFT |
|---|---|---|
| **`b-mc2/sql-create-context`** (78,577 rows) | The **narrow target task**: `question` + `context` (a literal `CREATE TABLE ...` schema) → `answer` (one canonical SQL string). | MTFT's target task must be **objectively scoreable**, or you cannot draw a frontier — and this dataset is built for that: exactly one correct output per row, so **exact match** works with no judge model and no ROUGE. The schema is a **separate field** from the question because the model must learn to condition on structure it did not author. It is also the source of this notebook's length asymmetry: completions average **~15 tokens**, roughly **13× shorter** than the chat anchor's — the concrete cause of the token-mean trap above. |
| **`HuggingFaceH4/no_robots`** (`train` + `test`) | The **anchor / rehearsal stream**: 10k human-written conversational instructions across 10 categories. | The anchor's job is to *represent the capability you are trying not to lose*, so it must match the distribution the model was originally tuned on — general, diverse, multi-category chat. Crucially it ships a **`test` split**, which is the never-trained held-out set used here to measure conversational forgetting. Its ~200-token completions are deliberately kept (not truncated to match SQL) because **the length imbalance is part of the problem being demonstrated**, not an artefact to hide. |
| **`SetFit/ag_news`** (**eval only**) | 4-way topic classification via rank classification. | The **general-capability probe**, and the important property is that **nothing in the training mixture teaches it** — not SQL, not chat. It therefore measures retained *latent* ability rather than anything either stream is optimising, which is what makes it a valid forgetting detector rather than a second target metric. |
| **`allenai/tulu-3-sft-mixture`** (939,343 rows) *(scale-up)* | The flagship open mixture, every row tagged with `source`. | What a production anchor stream actually looks like: dozens of sub-sources, each independently re-weightable, with the `source` column existing precisely so mixture ratios can be ablated. Swap it in for `no_robots` when you graduate from a 2-task demo to a real 10-stream blend. |

**Why this set of datasets and not one big blend:** MTFT's data contract is *deliberately asymmetric* — one stream you are optimising **for**, one or more streams you are optimising **against regression on**, and at least one eval set that **neither** stream trains on. If every dataset in your mixture is also a dataset you measure, you have no forgetting detector and no way to tell specialisation from collapse.

> This notebook fine-tunes **`Qwen/Qwen2.5-0.5B-Instruct`** (an already-conversational model — you cannot demonstrate forgetting on a model with nothing to forget) on **`b-mc2/sql-create-context`** blended with **`HuggingFaceH4/no_robots`** at **three mixture ratios**, and measures all three at once.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **Joint beats sequential because of what the gradient contains, not how much data you have.** Under sequential training the update direction is `∇L_target` alone; nothing in the optimisation problem references `L_general`, so any drift in that direction is free. Under MTFT the update is `Σ w_i ∇L_i`, so every single step carries a component that *resists* increasing the anchor task's loss. **You do not need much anchor data for this — you need it present in every batch.** That is why a small anchor fraction (5–30 %) recovers most of the retention.
- **Why the frontier is the deliverable.** The two objectives genuinely compete: at `p_target = 1.0` you get maximum target performance and maximum forgetting; at `p_target → 0` the reverse. There is no single "correct" ratio — there is a curve, and the right operating point is a **product decision** about how much general capability the deployment can afford to lose. Anyone who reports one MTFT number has not measured the thing that matters.
- **Why sampling ratio and loss weight must be separate arguments.** For the *expected mean* gradient, sampling task `i` twice as often and weighting it `2×` are equivalent. Under a real optimiser they are not:
  - **Adam** divides by a per-coordinate running second moment, so a **rarely-sampled, heavily-weighted** task produces high-variance, spiky updates that the moment estimates smear across subsequent steps.
  - **Gradient accumulation** makes it worse: a task that appears in 1 of 8 micro-batches contributes a burst, not a steady signal.
  - Practical rule: **use `p_i` to control presence, `w_i` only to correct a genuine scale mismatch** between objectives (e.g. one task's loss is naturally an order of magnitude larger).
- **Why per-example averaging, not per-token.** With token-level averaging the gradient share of a task is proportional to `p_i × (mean completion length of task i)`, not to `p_i`. For this notebook's streams (`SQL ≈ 15` tokens, `chat ≈ 200`), a **50/50 sampling ratio yields roughly a 7/93 gradient split** — the target task nearly vanishes. Averaging **within each example first, then across examples**, makes the configured ratio the realised ratio. Section 3 prints both numbers from a real batch so the gap is a measurement, not a claim.
- **Why `interleave_datasets(probabilities=...)` rather than `concatenate`.** Concatenation fixes the ratio at the *corpus* level and leaves ordering to the shuffle; interleaving samples from each stream with an explicit probability, which (a) makes the ratio a first-class parameter you can sweep, and (b) works identically for streaming datasets you cannot materialise. `stopping_strategy` then decides whether the small stream is **oversampled** (`all_exhausted`) or the large one **truncated** (`first_exhausted`) — an actual modelling choice, not a detail.
- **Why the total example count is held fixed across the sweep.** Different ratios naturally produce different mixture sizes, which would confound composition with step count. Every run below sees **exactly `MIX_SIZE` examples and the same number of optimizer steps**; only the blend changes. Without this the sweep measures nothing.

#### VRAM & Compute Impact

- **Per-step memory and FLOPs are identical to single-task SFT.** MTFT is not architecturally expensive — the loss is a weighted sum over the same batch. Two costs are real, and neither is VRAM:

  | Cost | Why | This notebook |
  |---|---|---|
  | **The sweep multiplies training cost by `n_ratios`** | The frontier *is* the method | **3 runs** × ~4 min |
  | **Eval multiplies by `n_axes`** | 1 target metric + `k` retention metrics, every run | **3 axes** × 4 models (baseline + 3) |
  | Custom `compute_loss` upcast | `logits.float()` for stable CE — `(B, L, V)` in fp32 | ~0.6 GB at `B=2, L=512, V=151936` |
  | Mixed-length batches | SQL ~150 tok next to chat ~500 tok → padding | mitigated by `group_by_length=True` |

- **Concrete T4 budget** (`Qwen2.5-0.5B-Instruct`, 4-bit NF4, `max_length=512`, batch 2 × accum 8, LoRA r=32):

  | Component | Cost |
  |---|---|
  | Base weights (4-bit NF4 + double quant) | ~0.40 GB |
  | LoRA r=32, all 7 projections × 24 layers | ~17.6 M params → ~0.2 GB with `paged_adamw_8bit` |
  | Activations, gradient-checkpointed | ~1.0–1.5 GB |
  | fp32 logits in the custom loss | ~0.6 GB |
  | **Peak** | **~3–4 GB** |
  | **Total wall clock (3 runs + 4 evaluations)** | **~20–25 min** |

- **A fresh model per run, not a reset adapter.** Reloading the 4-bit base costs ~15 s from the HF cache and gives provably independent runs; juggling named PEFT adapters and `requires_grad` across runs is a subtle-bug generator for no measurable gain.
- **Honest caveat about the size of the effect: LoRA forgets *less* by construction.** *Biderman et al., 2024* show low-rank updates both learn less and forget less than full fine-tuning — the same constraint does both. So the forgetting this notebook exhibits at `r=32` is the **mild** version. To see the textbook collapse, raise the learning rate, raise the rank, add epochs, or fine-tune fully. The mechanism and the measurement are identical; only the magnitude changes.

#### Pros & Cons

**Pros**
- **The only mechanism that makes retention an optimisation constraint** rather than a hope.
- **One model, many capabilities** — no inference-time adapter routing, and skills can compose within a single response.
- **Cheap in data.** A small anchor fraction present in every batch does most of the work; you do not need to re-train on the full original mixture.
- **Exposes negative transfer early** — if two tasks fight, the joint run shows it immediately in per-task loss curves.
- **The ratio is a real product dial.** "How much general capability may we trade for SQL accuracy?" becomes a number you can set, rather than an accident of the training script.
- **Composes with everything upstream** — CPT for knowledge, IT for the interface, MTFT to specialise without regressing.

**Cons**
- **`n_ratios ×` the training cost.** The frontier cannot be inferred from one run, so MTFT is inherently a sweep, and the sweep is the budget.
- **Mixture design is a large search space** with weak theory: `T` ratios, `T` weights, `T` masking policies, and no reliable way to predict the optimum without training.
- **Task interference / negative transfer is real** and can make joint training *worse* than sequential for a specific pair — the reason GradNorm, PCGrad and uncertainty weighting exist.
- **Every added task dilutes the target task's share** of a fixed compute budget; retention is not free, it is paid for in target-task performance.
- **Evaluation cost and complexity multiply.** You need a held-out set per axis, and axes frequently disagree.
- **The anchor stream is a proxy, not the truth.** You almost never have the model's real original training mixture, so you are rehearsing something *similar* and hoping it spans the capability you care about.
- **Length and difficulty imbalance silently distort the blend** unless you handle averaging explicitly — the trap this notebook measures.

#### Metrics to watch

- **Target-task metric** (SQL exact match here) — the thing you are buying.
- **≥ 1 retention metric on data that is in NEITHER training stream** (AG News rank classification here). If your only "general" metric is your anchor set, you are measuring memorisation of the anchor, not retained capability.
- **Held-out anchor-distribution perplexity** (no_robots `test`) — the fast, cheap forgetting signal, computed over completion tokens only.
- **Realised mixture composition** — count `task_id`s in the built mixture and assert it matches the configured `p`. Samplers do not always do what you asked.
- **Per-task loss curves, logged separately.** The aggregate loss can fall while one task's loss diverges; the average hides it completely.
- **Token-mean vs example-mean loss gap** — quantifies how far your realised gradient ratio has drifted from your configured sampling ratio.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit NF4) + `interleave_datasets(probabilities=...)` + a `task_id`-aware weighted loss, swept over three mixture ratios and scored on three axes.**

> ⚙️ **Why a custom `Trainer.compute_loss`:** MTFT's whole content is *how the per-task losses are combined*. HF's built-in CLM loss silently averages over all supervised tokens in the batch, which — with a 15-token target task and a 200-token anchor — turns a 50/50 sampling ratio into a ~7/93 gradient ratio. That is not a tuning detail, it is the algorithm being wrong. So the loss is written out, per-task weights are a first-class argument, and the gap against the built-in loss is **measured on a real batch** before any training happens.

**Executable pipeline:**

| Step | What | MTFT-specific detail |
|---|---|---|
| 1 | 4-bit **`Qwen2.5-0.5B-Instruct`** | **Instruct**, not base — you need capability *to lose* |
| 2 | Narrow target stream (SQL) + anchor stream (chat) + 3 eval sets | asymmetric by design |
| 3 | Completion-masked tokenization, carrying a **`task_id`** column | the id is what makes per-task anything possible |
| 4 | `interleave_datasets(probabilities=[p, 1-p])`, **fixed `MIX_SIZE`** | ratio is a parameter; step count is held constant |
| 5 | `TaskWeightedTrainer` + collator that preserves `task_id`; **example-mean** loss | plus a self-check vs. the built-in token-mean loss |
| 6 | Baseline evaluation of the untouched Instruct model | the row every other row is compared against |
| 7 | **Sweep** `p ∈ {1.00, 0.70, 0.50}` → the Pareto table | 3 independent runs, 3 metrics each |
| 8 | Load two adapters side by side and A/B them on SQL *and* chat | the failure made visible |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers datasets peft accelerate bitsandbytes

In [ ]:
import os, gc, math, re, time
from collections import Counter

import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset, interleave_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

set_seed(42)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# Turing (sm_75) has no bf16 tensor cores; forcing bf16 makes the bitsandbytes 4-bit dequant
# path return garbage, which surfaces as NaN loss or a run that trains to nothing.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

# ---- MTFT's knobs -----------------------------------------------------------------
MAX_LENGTH = 512    # SQL rows are ~150 tok, chat rows far longer — see the drop report in Step 3
MIX_SIZE = 800    # examples per run, HELD FIXED across the sweep so only composition varies
NARROW_POOL = 1500   # SQL examples available to sample from
ANCHOR_POOL = 2000   # chat examples available to sample from (many exceed MAX_LENGTH)
NUM_EPOCHS = 2

# The sweep: p = probability of drawing from the NARROW target stream.
# p=1.00 is the control — pure single-task fine-tuning, i.e. the forgetting baseline.
RATIOS = [1.00, 0.70, 0.50]

# Task ids are the spine of everything per-task: sampling reports, loss weights, loss curves.
TASK_NARROW, TASK_ANCHOR = 0, 1
TASK_NAMES = {TASK_NARROW: "sql(target)", TASK_ANCHOR: "chat(anchor)"}

# Per-task LOSS WEIGHTS — a DIFFERENT dial from the sampling ratio (see [Context Block]).
# Kept at 1.0 so the sweep isolates the effect of p alone; change one thing at a time.
TASK_WEIGHTS = {TASK_NARROW: 1.0, TASK_ANCHOR: 1.0}

### Step 1 — The **`-Instruct`** checkpoint (you cannot forget what you never knew)

The single most important setup decision in this notebook, and the opposite of the previous two:

- **CPT** started from a base model to install knowledge. **IT** started from a base model to install an interface.
- **MTFT starts from a model that already has the capability under threat.** Demonstrating "prevents catastrophic forgetting of general conversational capabilities" on a base model is impossible — there is no conversational capability there to lose. So: **`Qwen/Qwen2.5-0.5B-Instruct`**.
- In a real pipeline this checkpoint would be *your own* output from CPT → IT. `Qwen2.5-0.5B-Instruct` stands in for that, and it brings a working chat template plus a correct `eos_token` (`<|im_end|>`) — the Instruct checkpoint has none of the base model's EOS mismatch that the IT notebook had to repair.
- `load_4bit()` is a **factory**, not a single model: the sweep needs a provably fresh set of weights per run.

In [ ]:
# INSTRUCT, not base: MTFT protects an existing capability, so the model must have one.
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # fp16 on T4
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
# The Instruct checkpoint's eos IS the turn terminator (<|im_end|>) — no repair needed here,
# unlike the base checkpoint in the instruction-tuning notebook. Verify rather than assume.
print(f"eos={tokenizer.eos_token!r}({tokenizer.eos_token_id}) pad={tokenizer.pad_token!r}({tokenizer.pad_token_id})")
print("chat template present:", tokenizer.chat_template is not None)

def load_4bit():
    """Fresh 4-bit base. The sweep needs independent weights per run — see [Context Block]."""
    m = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa",   # FlashAttention-2 needs Ampere+
    )
    m.config.use_cache = False  # required with gradient checkpointing
    return m

def free_gpu(*names):
    """Drop references and actually give the memory back — 4 model lifetimes in one session."""
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    torch.cuda.empty_cache()

# LoRA sized for AGGRESSIVE narrow adaptation: r=32 (vs IT's 16) with a high LR is what makes
# forgetting visible at all on a T4. Low-rank updates forget less by construction, so a timid
# config would show a null result for the wrong reason.
peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = load_4bit()   # one plain model for the loss self-check and the baseline evaluation
print(f"\n{model.get_memory_footprint()/1e9:.2f} GB base (4-bit)")

### Step 2 — Two asymmetric streams, three eval sets

The data contract is **deliberately lopsided**, which is what distinguishes MTFT's data work from instruction tuning's:

| Stream / set | Source | Role |
|---|---|---|
| **Narrow target** | `b-mc2/sql-create-context` | optimise **for** it — scored by exact match |
| **Anchor** | `HuggingFaceH4/no_robots` `train` | optimise **against regression** — never scored directly |
| Target eval | held-out SQL rows | the target axis |
| Anchor eval | `no_robots` **`test`** | forgetting axis 1 (in-distribution for the anchor) |
| Capability probe | `SetFit/ag_news` | forgetting axis 2 — **in neither training stream** |

That last row is the one people skip. If every general metric comes from your anchor set, you are measuring how well you memorised the anchor, not whether the model retained anything.

In [ ]:
# ---- Narrow target stream: text-to-SQL --------------------------------------------
sql_raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)

SQL_INSTRUCTION = (
    "Given the database schema below, write a single SQL query that answers the question.\n\n"
    "Schema:\n{context}\n\nQuestion: {question}"
)

def sql_to_messages(ex):
    return {
        "messages": [
            {"content": SQL_INSTRUCTION.format(context=ex["context"], question=ex["question"]),
             "role": "user"},
            {"content": ex["answer"], "role": "assistant"},
        ],
        "task_id": TASK_NARROW,
    }

sql_all = sql_raw.select(range(NARROW_POOL + 60)).map(sql_to_messages, remove_columns=sql_raw.column_names)
sql_train = sql_all.select(range(NARROW_POOL))
# Keep the raw fields for exact-match scoring (the gold SQL string, not its tokenization).
sql_gold = sql_raw.select(range(NARROW_POOL, NARROW_POOL + 60))

print(f"SQL target stream: {len(sql_train)} train / {len(sql_gold)} eval")
print("\n--- sample target example ---")
print("USER     :", sql_train[0]["messages"][0]["content"][:220])
print("ASSISTANT:", sql_train[0]["messages"][1]["content"])

In [ ]:
# ---- Anchor stream: general conversation (the capability we refuse to lose) --------
nr_train = load_dataset("HuggingFaceH4/no_robots", split="train").shuffle(seed=42)
nr_test = load_dataset("HuggingFaceH4/no_robots", split="test").shuffle(seed=42)   # NEVER trained on

def nr_to_messages(ex):
    # Same key order as the SQL stream: interleave_datasets compares Arrow schemas exactly,
    # and list<struct<...>> field order is part of the schema.
    return {"messages": [{"content": m["content"], "role": m["role"]} for m in ex["messages"]],
            "task_id": TASK_ANCHOR}

def usable(ex):
    return len(ex["messages"]) >= 2 and ex["messages"][-1]["role"] == "assistant"

anchor_train = (nr_train.map(nr_to_messages, remove_columns=nr_train.column_names)
                        .filter(usable).select(range(ANCHOR_POOL)))
anchor_eval = (nr_test.map(nr_to_messages, remove_columns=nr_test.column_names)
                      .filter(usable).select(range(60)))

print(f"anchor stream: {len(anchor_train)} train / {len(anchor_eval)} held-out eval (no_robots TEST split)")

# ---- Capability probe: in NEITHER stream ------------------------------------------
ag_full = load_dataset("SetFit/ag_news", split="test")
AG_LABELS = sorted(set(ag_full["label_text"]))
ag_eval = ag_full.shuffle(seed=42).select(range(60))
print(f"capability probe: AG News 4-way topic classification, labels {AG_LABELS} (chance {100/len(AG_LABELS):.0f}%)")

### Step 3 — Completion-masked tokenization, **carrying `task_id`**

Same masking mechanics as the instruction-tuning notebook (render to text, tokenize the halves separately so the mask boundary is exact by construction), with one addition that everything else in this notebook depends on:

- **Each example keeps its `task_id`.** That integer is what makes per-task loss weights, per-task loss curves and realised-composition checks possible at all. Without it, a mixture is an undifferentiated pile of tokens and MTFT degenerates into ordinary SFT on a bigger dataset.
- The printout below reports **mean completion length per task** — the number that determines how badly token-level averaging would distort your configured ratio.

In [ ]:
def build_masked_example(messages, task_id):
    """Completion-only labels + the task_id that makes per-task anything possible."""
    prompt_msgs, answer = messages[:-1], messages[-1]

    # Render to TEXT, tokenize the halves SEPARATELY. Tokenizing the whole conversation and
    # slicing at len(prompt_ids) is unsafe: Qwen's pre-tokenizer merges newline runs, so a
    # completion starting with "\n" fuses with the template's trailing newline into one token
    # and every label past the boundary shifts. The string split is exact by construction.
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(prompt_msgs + [answer], tokenize=False)
    if not full_text.startswith(prompt_text):
        c = len(os.path.commonprefix([prompt_text, full_text]))
        raise ValueError(f"chat template not prefix-consistent at char {c}: "
                         f"{prompt_text[c:c+40]!r} vs {full_text[c:c+40]!r}")

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(full_text[len(prompt_text):], add_special_tokens=False)["input_ids"]
    if len(prompt_ids) + len(completion_ids) > MAX_LENGTH:
        return None                     # DROP, never truncate: a cut completion teaches "don't stop"

    return {
        "input_ids": prompt_ids + completion_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(completion_ids)),
        "labels": [-100] * len(prompt_ids) + completion_ids,   # completion-only supervision
        "task_id": task_id,   # <-- the MTFT-specific column
    }

def tokenize_stream(ds, desc):
    rows, dropped = [], 0
    for ex in ds:
        out = build_masked_example(ex["messages"], ex["task_id"])
        if out is None:
            dropped += 1
        else:
            rows.append(out)
    comp = [sum(1 for l in r["labels"] if l != -100) for r in rows]
    print(f"{desc:<14} kept {len(rows):>4}, dropped {dropped:>4} over {MAX_LENGTH} tok | "
          f"mean completion {sum(comp)/max(1,len(comp)):6.1f} tok")
    return Dataset.from_list(rows), sum(comp) / max(1, len(comp))

narrow_tok, narrow_meanlen = tokenize_stream(sql_train, "sql(target)")
anchor_tok, anchor_meanlen = tokenize_stream(anchor_train, "chat(anchor)")
anchor_eval_tok, _ = tokenize_stream(anchor_eval, "chat eval")

assert len(anchor_tok) >= 400, (
    f"only {len(anchor_tok)} anchor examples survived MAX_LENGTH={MAX_LENGTH}; "
    "raise MAX_LENGTH or ANCHOR_POOL")

ratio = anchor_meanlen / max(1e-9, narrow_meanlen)
print(f"\nlength asymmetry: anchor completions are {ratio:.1f}x longer than target completions")
print(f"=> under TOKEN-mean averaging, a 50/50 sampling ratio becomes a "
      f"{100*narrow_meanlen/(narrow_meanlen+anchor_meanlen):.0f}/"
      f"{100*anchor_meanlen/(narrow_meanlen+anchor_meanlen):.0f} GRADIENT ratio. Step 5 measures this.")

### Step 4 — The mixture: **`interleave_datasets(probabilities=...)`** at a fixed size

- **`probabilities=[p, 1-p]`** makes the ratio an explicit, sweepable parameter — unlike concatenation, which bakes the ratio into corpus sizes and leaves the rest to the shuffle. It also works unchanged on streaming datasets you cannot materialise.
- **`stopping_strategy="all_exhausted"`** *oversamples* the smaller stream (repeating examples) rather than truncating the larger one, so a small anchor fraction can still be drawn from throughout the run.
- **`.select(range(MIX_SIZE))` is what makes the sweep valid.** Different ratios yield different natural mixture sizes; fixing the count means every run gets the same number of optimizer steps and only the **composition** varies. Without it you would be measuring ratio *and* training length at once.
- **`p = 1.0` is special-cased** to the pure target stream — it is the single-task control, and `probabilities=[1.0, 0.0]` is a degenerate sampler argument, not a mixture.
- The realised composition is **counted and asserted**, because a sampler that quietly does something else invalidates every number downstream.

In [ ]:
def make_mixture(p_narrow, size=MIX_SIZE, seed=42):
    """Blend the two streams at an explicit ratio, fixed total size.

    p_narrow = 1.0 is the single-task control (pure narrow fine-tuning, the forgetting baseline).
    """
    if p_narrow >= 1.0:
        return narrow_tok.shuffle(seed=seed).select(range(min(size, len(narrow_tok))))

    mixed = interleave_datasets(
        [narrow_tok, anchor_tok],
        probabilities=[p_narrow, 1.0 - p_narrow],
        seed=seed,
        # all_exhausted: OVERSAMPLE the smaller stream instead of truncating the larger one,
        # so the anchor keeps appearing across the whole run rather than only early on.
        stopping_strategy="all_exhausted",
    )
    return mixed.select(range(min(size, len(mixed))))

# Verify the sampler actually produced the ratio we asked for, before spending GPU time on it.
for p in RATIOS:
    mix = make_mixture(p)
    counts = Counter(mix["task_id"])
    realised = counts[TASK_NARROW] / len(mix)
    print(f"p_narrow={p:.2f} -> {len(mix)} examples | realised {realised:.2f} "
          f"({ {TASK_NAMES[k]: v for k, v in counts.items()} })")
    assert abs(realised - p) < 0.08, f"sampler drift: asked {p}, got {realised:.3f}"
print("\nrealised composition matches the configured ratios")

### Step 5 — `TaskWeightedTrainer`: per-task weights and **example-mean** averaging

The heart of the notebook. Three things the built-in loss cannot do:

1. **Average within each example first, then across examples.** HF's CLM loss is a mean over *all supervised tokens in the batch*, which weights each example by its completion length. With a 15-token target and a ~200-token anchor, that alone rewrites your mixture ratio (Step 3 quantified it). Example-mean makes `p` mean what you configured.
2. **Apply a per-task loss weight `w_i`** — the second, independent dial from [Context Block]. Kept at 1.0 in this sweep so exactly one variable moves.
3. **Log per-task losses separately.** The aggregate can fall while one task diverges; the average hides that completely.

The collator needs a small wrapper because `task_id` is not a sequence to be padded — it is popped, the rest is collated normally by `DataCollatorForSeq2Seq`, then it is re-attached as a tensor. `remove_unused_columns=False` keeps it alive that far.

The cell after this one is a **self-check against the built-in loss on a real batch** — it prints both numbers, so the token-mean/example-mean gap is a measurement rather than an assertion.

In [ ]:
class TaskAwareCollator:
    """DataCollatorForSeq2Seq (pads input_ids with pad_token_id AND labels with -100),
    plus passthrough of the non-sequence `task_id` column as a tensor."""

    def __init__(self, tokenizer):
        self.inner = DataCollatorForSeq2Seq(
            tokenizer=tokenizer, model=None, padding=True,
            label_pad_token_id=-100, pad_to_multiple_of=8,
        )

    def __call__(self, features):
        # Read, don't pop: mutating caller-owned dicts breaks any second call on the same rows.
        task_ids = [f["task_id"] for f in features]       # not a sequence — must not be padded
        batch = self.inner([{k: v for k, v in f.items() if k != "task_id"} for f in features])
        batch["task_id"] = torch.tensor(task_ids, dtype=torch.long)
        return batch


class TaskWeightedTrainer(Trainer):
    """Weighted-sum-of-tasks loss with per-EXAMPLE averaging and per-task logging."""

    def __init__(self, *args, task_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # transformers>=4.46 skips its `loss /= gradient_accumulation_steps` step when the model
        # accepts loss kwargs and a custom compute_loss returns an already-reduced loss. Left
        # alone that makes the effective learning rate 8x too high here. Force the classic path.
        self.model_accepts_loss_kwargs = False
        n = max(task_weights) + 1
        self.weight_vec = torch.tensor([task_weights.get(i, 1.0) for i in range(n)])
        self.task_loss_sum, self.task_loss_count = Counter(), Counter()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        task_id = inputs.pop("task_id")
        labels = inputs.pop("labels")

        # No `labels=` to the model: we compute the loss ourselves, so letting the model also
        # compute its (token-mean) loss would just burn a second cross-entropy.
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

        # Causal shift: position t predicts token t+1.
        shift_logits = outputs.logits[:, :-1, :].float()   # fp32 for stable CE (see VRAM table)
        shift_labels = labels[:, 1:]

        per_token = F.cross_entropy(
            shift_logits.transpose(1, 2), shift_labels, ignore_index=-100, reduction="none",
        )                                                   # (B, L-1), zeros at ignored positions
        supervised = (shift_labels != -100)
        # THE key line: mean WITHIN each example, so a long completion does not outvote a short
        # one. HF's built-in loss instead means over all tokens in the batch, which silently
        # re-weights the mixture by completion length.
        per_example = (per_token * supervised).sum(1) / supervised.sum(1).clamp(min=1)

        w = self.weight_vec.to(per_example.device)[task_id]
        loss = (per_example * w).mean()

        # Per-task bookkeeping — the aggregate loss cannot show a single task diverging.
        for t, l in zip(task_id.tolist(), per_example.detach().tolist()):
            self.task_loss_sum[t] += l
            self.task_loss_count[t] += 1

        return (loss, outputs) if return_outputs else loss

    def per_task_losses(self):
        return {TASK_NAMES[t]: self.task_loss_sum[t] / self.task_loss_count[t]
                for t in sorted(self.task_loss_count)}


collator = TaskAwareCollator(tokenizer)
print("collator + trainer ready")

In [ ]:
# ---- Self-check: our example-mean loss vs HF's built-in token-mean loss -----------
# Same batch, same weights (all 1.0). If these two numbers differ, the difference IS the
# hidden re-weighting that token-level averaging applies to a mixed-length mixture.
probe_rows = [narrow_tok[i] for i in range(2)] + [anchor_tok[i] for i in range(2)]
probe = collator([dict(r) for r in probe_rows])
probe = {k: v.to(model.device) for k, v in probe.items()}

with torch.no_grad():
    labels = probe["labels"]
    out = model(input_ids=probe["input_ids"], attention_mask=probe["attention_mask"])
    sl, tl = out.logits[:, :-1, :].float(), labels[:, 1:]
    per_token = F.cross_entropy(sl.transpose(1, 2), tl, ignore_index=-100, reduction="none")
    sup = (tl != -100)
    per_example = (per_token * sup).sum(1) / sup.sum(1).clamp(min=1)

    ours = per_example.mean().item()  # EXAMPLE-mean (this notebook)
    hf_builtin = (per_token * sup).sum().item() / sup.sum().item()  # TOKEN-mean (HF default)

    # Cross-check that our per-token surface matches the model's own loss implementation.
    ref = model(input_ids=probe["input_ids"], attention_mask=probe["attention_mask"],
                labels=labels).loss.item()

print(f"per-example losses: {[round(x, 3) for x in per_example.tolist()]}")
print(f"supervised tokens/row: {sup.sum(1).tolist()}   <- the imbalance that does the damage")
print(f"EXAMPLE-mean (ours): {ours:.4f}")
print(f"TOKEN-mean   (HF): {hf_builtin:.4f}")
print(f"model's own .loss: {ref:.4f}   (sanity: should match TOKEN-mean)")
assert abs(hf_builtin - ref) < 0.05, "our per-token CE disagrees with the model's own loss"
print(f"\ngap: {100*abs(ours-hf_builtin)/hf_builtin:.1f}% — with all weights at 1.0 the two "
      f"averagings already disagree, purely because of completion-length imbalance.")

### Step 6 — The three evaluation axes

One target metric, two retention metrics, all applied to **every** model in the sweep including the untouched baseline:

- **`sql_exact_match`** — greedy generation, normalised string comparison (lowercased, whitespace collapsed, trailing `;` stripped). No judge, no ROUGE; this is why the target task was chosen for having one canonical answer.
- **`anchor_ppl`** — perplexity over **completion tokens only** on `no_robots` **`test`**. The cheap, in-distribution forgetting signal.
- **`ag_accuracy`** — rank classification on AG News: score each label as a continuation, take the argmax. **Neither training stream teaches this**, which is exactly what makes it a forgetting detector rather than a second target metric.

In [ ]:
def _norm_sql(s):
    return re.sub(r"\s+", " ", s.strip().lower()).rstrip(";")

@torch.no_grad()
def sql_exact_match(m, n=40, max_new_tokens=64):
    """Target-task metric: greedy decode, normalised exact match against the gold SQL."""
    m.eval()
    hits = 0
    for i in range(n):
        ex = sql_gold[i]
        prompt = SQL_INSTRUCTION.format(context=ex["context"], question=ex["question"])
        text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(m.device)
        out = m.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                         eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
        pred = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        hits += _norm_sql(pred) == _norm_sql(ex["answer"])
    return 100 * hits / n

@torch.no_grad()
def anchor_ppl(m, ds=None, batch_size=2, max_rows=40):
    """Forgetting axis 1: perplexity over COMPLETION tokens of held-out anchor conversations."""
    ds = anchor_eval_tok if ds is None else ds
    m.eval()
    nll, ntok = 0.0, 0
    rows = [dict(ds[i]) for i in range(min(len(ds), max_rows))]
    for i in range(0, len(rows), batch_size):
        batch = collator(rows[i:i + batch_size])
        batch.pop("task_id")
        batch = {k: v.to(m.device) for k, v in batch.items()}
        labels = batch.pop("labels")
        logits = m(**batch).logits[:, :-1, :].float()
        tl = labels[:, 1:]
        pt = F.cross_entropy(logits.transpose(1, 2), tl, ignore_index=-100, reduction="none")
        sup = (tl != -100)
        nll += (pt * sup).sum().item()
        ntok += sup.sum().item()
    return math.exp(nll / max(1, ntok))

AG_TEMPLATE = ("Classify the topic of the following news article. "
               "Answer with one of: {opts}.\n\nArticle: {text}\n\nTopic:")

@torch.no_grad()
def ag_accuracy(m, ds=None, labels=None):
    """Forgetting axis 2: rank classification on a task NEITHER stream trains on."""
    ds, labels = (ag_eval if ds is None else ds), (AG_LABELS if labels is None else labels)
    m.eval()
    hits = 0
    for ex in ds:
        prompt = AG_TEMPLATE.format(opts=", ".join(labels), text=ex["text"][:1000])
        text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                             tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        scores = []
        for lab in labels:
            lab_ids = tokenizer(lab, add_special_tokens=False)["input_ids"]
            ids = torch.tensor([prompt_ids + lab_ids], device=m.device)
            lp = m(input_ids=ids).logits[0][len(prompt_ids) - 1:-1].float().log_softmax(-1)
            pos = torch.arange(len(lab_ids), device=lp.device)
            scores.append(lp[pos, torch.tensor(lab_ids, device=lp.device)].mean().item())
        hits += labels[max(range(len(labels)), key=lambda j: scores[j])] == ex["label_text"]
    return 100 * hits / len(ds)

def evaluate_all(m, tag):
    t0 = time.time()
    res = {"sql_em": sql_exact_match(m), "anchor_ppl": anchor_ppl(m), "ag_acc": ag_accuracy(m)}
    print(f"[{tag:<18}] SQL EM {res['sql_em']:5.1f}% | anchor PPL {res['anchor_ppl']:7.2f} | "
          f"AG News {res['ag_acc']:5.1f}%   ({time.time()-t0:.0f}s)")
    return res

In [ ]:
# ---- The reference row: the untouched Instruct model ------------------------------
# Every number in the sweep is only meaningful relative to this. Expect near-zero SQL EM
# (it has never seen the task) and the HIGHEST general scores it will ever have.
model.config.use_cache = True     # load_4bit() disables it for training; generation wants it
baseline = evaluate_all(model, "instruct baseline")

# The plain model has done its two jobs (loss self-check + baseline). Reclaim its VRAM before
# the sweep starts building models of its own.
free_gpu("model", "out", "probe", "sl", "tl", "sup", "per_token", "per_example", "labels")
free, total = torch.cuda.mem_get_info()
print(f"GPU free before sweep: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

### Step 7 — The sweep: three ratios, three independent runs

Everything except `p_narrow` is held constant — same LoRA config, same LR, same `MIX_SIZE`, same step count, same seed, fresh weights each time. `p = 1.00` is the **control**: pure single-task fine-tuning, i.e. exactly the thing MTFT exists to fix.

**`learning_rate=3e-4` and 2 epochs is deliberately aggressive.** Low-rank adapters forget less by construction (*Biderman et al., 2024*), so a gentle configuration would produce a null result for the wrong reason. This is the "heavily adapting a model to a specific, narrow task" regime the concept describes.

In [ ]:
def run_mtft(p_narrow, task_weights=TASK_WEIGHTS, tag=None):
    """One independent MTFT run at mixture ratio p_narrow. Returns metrics + adapter path."""
    tag = tag or f"p={p_narrow:.2f}"
    mix = make_mixture(p_narrow)
    counts = Counter(mix["task_id"])

    m = get_peft_model(
        prepare_model_for_kbit_training(
            load_4bit(), use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        ),
        peft_config,
    )

    steps = math.ceil(len(mix) * NUM_EPOCHS / (2 * 8))
    out_dir = f"./mtft_p{int(round(p_narrow*100))}"

    args = TrainingArguments(
        output_dir=out_dir,
        run_name=f"mtft-{tag}",
        num_train_epochs=NUM_EPOCHS,
        learning_rate=3e-4,  # aggressive on purpose: forgetting must be visible
        lr_scheduler_type="cosine",
        warmup_steps=max(5, int(0.03 * steps)),
        max_grad_norm=1.0,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=USE_BF16, fp16=not USE_BF16,
        optim="paged_adamw_8bit",
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        label_names=["labels"],
        remove_unused_columns=False,  # REQUIRED: keeps `task_id` alive for the collator
        seed=42,
    )

    trainer = TaskWeightedTrainer(
        model=m, args=args, train_dataset=mix,
        data_collator=collator, processing_class=tokenizer,
        task_weights=task_weights,
    )

    print(f"\n{'='*74}\nRUN {tag} | {len(mix)} examples "
          f"({ {TASK_NAMES[k]: v for k, v in counts.items()} }) | {steps} optimizer steps\n{'='*74}")
    t0 = time.time()
    trainer.train()
    print(f"trained in {(time.time()-t0)/60:.1f} min | per-task mean loss: "
          + " | ".join(f"{k} {v:.4f}" for k, v in trainer.per_task_losses().items()))

    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    # Training leaves use_cache=False + checkpointing on; generation without a KV cache
    # recomputes the whole prefix per token (~4x slower for a 64-token decode).
    m.gradient_checkpointing_disable()
    m.config.use_cache = True

    res = evaluate_all(m, tag)
    res.update(p_narrow=p_narrow, adapter=out_dir,
               n_narrow=counts[TASK_NARROW], n_anchor=counts[TASK_ANCHOR],
               peak_gb=torch.cuda.max_memory_allocated() / 1e9)
    # `trainer`/`m` are LOCALS — free_gpu() reaches globals() only, so delete them here or the
    # next run in the sweep loads a second base model on top of this one's optimizer state.
    del trainer, m
    gc.collect()
    torch.cuda.empty_cache()
    return res

In [ ]:
results = []
torch.cuda.reset_peak_memory_stats()
for p in RATIOS:
    results.append(run_mtft(p))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free after run: {free/1e9:.2f} GB / {total/1e9:.2f} GB")
print("\nsweep complete")

In [ ]:
# ---- The Pareto frontier: target gain vs retained general capability -------------
print(f"{'run':<14}{'p_narrow':>9}{'SQL EM':>9}{'AG News':>9}{'anchorPPL':>11}"
      f"{'ΔAG':>8}{'ΔPPL%':>8}")
print("-" * 68)
print(f"{'baseline':<14}{'—':>9}{baseline['sql_em']:>8.1f}%{baseline['ag_acc']:>8.1f}%"
      f"{baseline['anchor_ppl']:>11.2f}{'—':>8}{'—':>8}")
for r in results:
    d_ag = r["ag_acc"] - baseline["ag_acc"]
    d_ppl = 100 * (r["anchor_ppl"] - baseline["anchor_ppl"]) / baseline["anchor_ppl"]
    label = "pure narrow" if r["p_narrow"] >= 1.0 else "MTFT blend"
    print(f"{label:<14}{r['p_narrow']:>9.2f}{r['sql_em']:>8.1f}%{r['ag_acc']:>8.1f}%"
          f"{r['anchor_ppl']:>11.2f}{d_ag:>+8.1f}{d_ppl:>+8.1f}")

pure = next(r for r in results if r["p_narrow"] >= 1.0)
blends = [r for r in results if r["p_narrow"] < 1.0]
if blends:
    best = max(blends, key=lambda r: r["ag_acc"])
    print(f"\nFORGETTING TAX of pure single-task tuning : "
          f"AG News {pure['ag_acc']-baseline['ag_acc']:+.1f} pts, "
          f"anchor PPL {100*(pure['anchor_ppl']-baseline['anchor_ppl'])/baseline['anchor_ppl']:+.1f}%")
    print(f"BEST-RETENTION BLEND (p={best['p_narrow']:.2f})        : "
          f"AG News {best['ag_acc']-baseline['ag_acc']:+.1f} pts, "
          f"anchor PPL {100*(best['anchor_ppl']-baseline['anchor_ppl'])/baseline['anchor_ppl']:+.1f}%, "
          f"at a target-task cost of {best['sql_em']-pure['sql_em']:+.1f} pts SQL EM")
    print("\nThat last line IS the Pareto trade-off. There is no 'best' row — only an operating")
    print("point you choose, given how much general capability the deployment can afford to lose.")
print(f"\npeak VRAM across the sweep: {max(r['peak_gb'] for r in results):.2f} GB")

---

## **[Key Observations]**

*MTFT is never reported as one number. Fill in the frontier, then state the operating point you chose and why.*

### Sweep configuration

| Setting | Value |
|---|---|
| Start checkpoint | `Qwen/Qwen2.5-0.5B-Instruct` (**already conversational**) |
| Target stream / anchor stream | `b-mc2/sql-create-context` / `HuggingFaceH4/no_robots` |
| `MIX_SIZE` (held fixed) · epochs · LR | |
| LoRA `r` / `alpha` / targets | |
| `TASK_WEIGHTS` | |
| Mean completion length: target vs anchor | ___ vs ___ tok (asymmetry ___×) |
| Realised composition per ratio | |

### The frontier

| Run | `p_narrow` | SQL EM ↑ | AG News ↑ | anchor PPL ↓ | ΔAG vs base | ΔPPL vs base |
|---|---|---|---|---|---|---|
| baseline (no training) | — | | | | — | — |
| pure narrow (control) | 1.00 | | | | | |
| MTFT blend | 0.70 | | | | | |
| MTFT blend | 0.50 | | | | | |

- **Chosen operating point:** `p = ___`, because ___
- **Forgetting tax of the control:** ___ pts AG News, ___ % anchor PPL
- **Target-task cost of retention:** ___ pts SQL EM

### Loss mechanics

| Metric | Value |
|---|---|
| Example-mean loss (ours) vs token-mean (HF), same batch | ___ vs ___ (gap ___ %) |
| Per-task mean loss at end of run: target / anchor | |
| Did either task's loss diverge while the aggregate fell? | |
| Peak VRAM (GB) · total sweep wall clock (min) | |

### Qualitative (Step 8)

- On a **SQL** prompt: does the pure-narrow adapter beat the blend, and by how much?
- On a **chat** prompt: does the pure-narrow adapter emit SQL, refuse to converse, or fail to terminate?
- Does the blend still sound like the original Instruct model, or has its register shifted?

### Ablations worth the compute

- **`p ∈ {0.9, 0.8, 0.3, 0.1}`** — fill in the curve; the interesting region is usually the knee, not the endpoints.
- **`TASK_WEIGHTS` vs `p`** — hold `p=0.5` and set `w_target=2.0`. Under Adam this is *not* the same as `p=0.67` at `w=1.0`; measure the difference.
- **Token-mean vs example-mean** — swap `per_example.mean()` for the token-mean and re-run `p=0.5`. Predicted: it behaves like a much higher `p`, because the short target completions get outvoted.
- **Full fine-tune instead of LoRA** — the forgetting should be dramatically larger. LoRA's low-rank constraint is itself a mild forgetting mitigation.
- **A third stream** (e.g. `allenai/tulu-3-sft-mixture`) — does retention improve per unit of target-task cost, or does the target task just get diluted?

## Export — Download the Adapters (Optional)

In [ ]:
import shutil, os

# Every run's adapter is on disk; zip the whole sweep so the frontier is reproducible.
output_filename = "mtft_sweep_adapters.zip"
staging = "./mtft_sweep"
os.makedirs(staging, exist_ok=True)
for r in results:
    dst = os.path.join(staging, os.path.basename(r["adapter"]))
    if os.path.isdir(r["adapter"]) and not os.path.isdir(dst):
        shutil.copytree(r["adapter"], dst)

shutil.make_archive(output_filename.replace(".zip", ""), "zip", staging)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
    print("contains:", sorted(os.listdir(staging)))
else:
    print("Zip not found — run the sweep first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did the sweep + zipping finish?")

---

## Step 8 — Model Usage: the forgetting, made visible

Numbers in a table are one thing; watching a model answer "what should I cook tonight?" with a `SELECT` statement is another.

**Both adapters are loaded onto one base model** as *named adapters* — `PeftModel.from_pretrained(..., adapter_name=...)` plus `load_adapter`, then `set_adapter` to switch. One base in VRAM, three-way A/B (base ↔ pure-narrow ↔ blend) with no reloading. This is also how you would ship several task adapters behind one served model.

Each pair of prompts is the whole thesis: **the target task, where specialising should win**, and **a general conversational turn, where the control should visibly break**.

In [ ]:
free_gpu()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

pure_path = f"./mtft_p{int(round(max(RATIOS)*100))}"          # p=1.00, the single-task control
blend_path = f"./mtft_p{int(round(min(RATIOS)*100))}"         # the most-blended run

infer_base = load_4bit()
infer_base.config.use_cache = True                            # KV cache back on for generation

# Two named adapters on ONE base: switch with set_adapter, no second copy of the weights.
infer = PeftModel.from_pretrained(infer_base, pure_path, adapter_name="pure")
infer.load_adapter(blend_path, adapter_name="blend")
infer.eval()
print("adapters loaded:", list(infer.peft_config.keys()))

@torch.no_grad()
def answer(instruction, which, max_new_tokens=96):
    """which: 'base' (adapters disabled) | 'pure' | 'blend'."""
    text = tokenizer.apply_chat_template([{"role": "user", "content": instruction}],
                                         tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(infer.device)

    def _gen():
        out = infer.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=1.05,
                             eos_token_id=tokenizer.eos_token_id,
                             pad_token_id=tokenizer.pad_token_id)
        txt = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        return txt.strip().replace("\n", " ")[:260]

    if which == "base":
        with infer.disable_adapter():
            return _gen()
    infer.set_adapter(which)
    return _gen()

In [ ]:
sql_probe = SQL_INSTRUCTION.format(
    context="CREATE TABLE employees (name VARCHAR, department VARCHAR, salary INTEGER)",
    question="What is the average salary in the engineering department?",
)
chat_probes = [
    "I have chicken, rice and a lemon. What should I cook tonight?",
    "Explain in two sentences why the sky is blue.",
]

print("### TARGET TASK — specialising should WIN here\n")
for which in ("base", "pure", "blend"):
    print(f"[{which:>5}] {answer(sql_probe, which)}")

print("\n\n### GENERAL CONVERSATION — the control should visibly BREAK here\n")
for p in chat_probes:
    print(f"--- {p}")
    for which in ("base", "pure", "blend"):
        print(f"[{which:>5}] {answer(p, which)}")
    print()